# v1.0 Capstone: CLI, Artifact Contract, and Module Map

## Notebook Contract
- **Objective:** the v1.0 release walkthrough. Lists every CLI command in one place, demonstrates the stable artifact contract via `verify_artifact`, and prints the module map so the rest of the lab is discoverable from a single notebook.
- **Inputs:** the installed `hf_finetuning_lab` package; a tiny ad-hoc artifact directory built in a temp folder.
- **Outputs:** the CLI surface, an artifact verification report, and a module map. Optionally writes the report to `reports/sample_run/v1_capstone/artifact_report.txt`.
- **Expected runtime:** under 15 seconds on CPU.
- **Scope boundary:** the notebook is read-only with respect to the package — it surfaces existing functionality rather than introducing new code.

## 1) Setup

In [1]:
from __future__ import annotations

import importlib
import platform
import sys
import tempfile
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path('..').resolve()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import pandas as pd
from typer.testing import CliRunner

from hf_finetuning_lab import __version__
from hf_finetuning_lab.artifacts import (
    ALTERNATIVE_REQUIRED_FILES,
    RECOMMENDED_FILES,
    REQUIRED_FILES,
    verify_artifact,
)
from hf_finetuning_lab.cli import app

print(f'Package version: {__version__}')
print(f'Python:          {sys.version.split()[0]}')
print(f'Platform:        {platform.platform()}')
print(f'Timestamp (UTC): {datetime.now(UTC).isoformat(timespec="seconds")}')

Package version: 1.0.0
Python:          3.12.11
Platform:        Windows-11-10.0.26200-SP0
Timestamp (UTC): 2026-05-21T09:17:38+00:00


## 2) Stable CLI Surface

`hf-lab list-commands` is the source of truth for the public CLI. Anything not on this list is internal.

In [2]:
runner = CliRunner()

result = runner.invoke(app, ['list-commands'])
commands = [line.strip() for line in result.output.splitlines() if line.strip()]
command_table = pd.DataFrame(
    {
        'command': commands,
        'invocation': [f'poetry run hf-lab {name}' for name in commands],
    }
)
command_table

,command,invocation
0,evaluate,poetry run hf-lab evaluate
1,fetch-hub-dataset,poetry run hf-lab fetch-hub-dataset
2,list-commands,poetry run hf-lab list-commands
3,list-hub-datasets,poetry run hf-lab list-hub-datasets
4,predict,poetry run hf-lab predict
5,sample-data,poetry run hf-lab sample-data
6,serve,poetry run hf-lab serve
7,train,poetry run hf-lab train
8,verify-artifact,poetry run hf-lab verify-artifact
9,version,poetry run hf-lab version


In [3]:
version_result = runner.invoke(app, ['version'])
print(f'CLI exit code: {version_result.exit_code}')
print(f'CLI output:    {version_result.output.strip()!r}')
assert version_result.output.strip() == __version__

CLI exit code: 0
CLI output:    '1.0.0'


## 3) Stable Artifact Contract

A model directory produced by `hf-lab train` follows a fixed layout. `verify_artifact` (and the `hf-lab verify-artifact` CLI command) walk the directory and report each slot, so CI can fail fast on broken artifacts.

In [4]:
spec_rows: list[dict[str, str]] = []
for name in REQUIRED_FILES:
    spec_rows.append({'tier': 'required', 'files': name})
for group in ALTERNATIVE_REQUIRED_FILES:
    spec_rows.append({'tier': 'required (alternatives)', 'files': ' | '.join(group)})
for name in RECOMMENDED_FILES:
    spec_rows.append({'tier': 'recommended', 'files': name})
pd.DataFrame(spec_rows)

,tier,files
0,required,config.json
1,required (alternatives),model.safetensors | pytorch_model.bin
2,required (alternatives),tokenizer.json | vocab.txt | vocab.json
3,recommended,tokenizer_config.json
4,recommended,special_tokens_map.json
5,recommended,model_card.md
6,recommended,metrics.json


In [5]:
# Build a minimal artifact in a temp directory so we never depend on a real
# trained model existing on the host. Everything is a tiny placeholder file —
# verify_artifact only inspects the layout, not the contents.
scratch = Path(tempfile.mkdtemp(prefix='hf-lab-capstone-'))
(scratch / 'config.json').write_text('{}', encoding='utf-8')
(scratch / 'model.safetensors').write_text('placeholder', encoding='utf-8')
(scratch / 'tokenizer.json').write_text('{}', encoding='utf-8')
(scratch / 'tokenizer_config.json').write_text('{}', encoding='utf-8')
(scratch / 'special_tokens_map.json').write_text('{}', encoding='utf-8')
(scratch / 'model_card.md').write_text('# Demo model card\n', encoding='utf-8')
(scratch / 'metrics.json').write_text('{}', encoding='utf-8')
print(f'Synthetic artifact at: {scratch}')
sorted(p.name for p in scratch.iterdir())

Synthetic artifact at: C:\Users\diogo\AppData\Local\Temp\hf-lab-capstone-iolwc_tq


['config.json',
 'metrics.json',
 'model.safetensors',
 'model_card.md',
 'special_tokens_map.json',
 'tokenizer.json',
 'tokenizer_config.json']

In [6]:
report = verify_artifact(scratch)
report_rows = [
    {'status': check.status.upper(), 'name': check.name, 'detail': check.detail}
    for check in report.checks
]
report_df = pd.DataFrame(report_rows)
print(f'Overall ok? {report.ok}')
print(f'Missing:    {report.missing}')
print(f'Warnings:   {report.warnings}')
report_df

Overall ok? True
Missing:    []
Warnings:   []


,status,name,detail
0,OK,config.json,present
1,OK,model.safetensors | pytorch_model.bin,present as `model.safetensors`
2,OK,tokenizer.json | vocab.txt | vocab.json,present as `tokenizer.json`
3,OK,tokenizer_config.json,present
4,OK,special_tokens_map.json,present
5,OK,model_card.md,present
6,OK,metrics.json,present


In [7]:
# Same check, but through the CLI surface.
cli_result = runner.invoke(app, ['verify-artifact', '--model-dir', str(scratch), '--strict'])
print(f'CLI exit code: {cli_result.exit_code}')
print(cli_result.output)

CLI exit code: 0
[OK ] config.json: present
[OK ] model.safetensors | pytorch_model.bin: present as `model.safetensors`
[OK ] tokenizer.json | vocab.txt | vocab.json: present as `tokenizer.json`
[OK ] tokenizer_config.json: present
[OK ] special_tokens_map.json: present
[OK ] model_card.md: present
[OK ] metrics.json: present



In [8]:
report_dir = ROOT / 'reports' / 'sample_run' / 'v1_capstone'
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / 'artifact_report.txt'
lines = [f'verify_artifact report for: {scratch}', '']
for check in report.checks:
    lines.append(f'[{check.status.upper():<7}] {check.name}: {check.detail}')
report_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'Report written to: {report_path}')

Report written to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\v1_capstone\artifact_report.txt


## 4) Module Map

Every top-level subpackage and the docstring it documents itself with. Use this as the table of contents when navigating the codebase or onboarding a new contributor.

In [9]:
modules = [
    'hf_finetuning_lab.cli',
    'hf_finetuning_lab.artifacts',
    'hf_finetuning_lab.config',
    'hf_finetuning_lab.data',
    'hf_finetuning_lab.data.hub',
    'hf_finetuning_lab.evaluation',
    'hf_finetuning_lab.evaluation.metrics',
    'hf_finetuning_lab.evaluation.robust',
    'hf_finetuning_lab.experiments',
    'hf_finetuning_lab.token_classification',
    'hf_finetuning_lab.retrieval',
    'hf_finetuning_lab.model_cards',
    'hf_finetuning_lab.governance',
    'hf_finetuning_lab.serving',
    'hf_finetuning_lab.sample_data',
]
rows: list[dict[str, str]] = []
for name in modules:
    try:
        module = importlib.import_module(name)
    except ImportError as exc:
        rows.append({'module': name, 'summary': f'(import failed: {exc})'})
        continue
    doc = (module.__doc__ or '').strip().splitlines()
    summary = doc[0] if doc else ''
    rows.append({'module': name, 'summary': summary})
pd.DataFrame(rows)

,module,summary
0,hf_finetuning_lab.cli,
1,hf_finetuning_lab.artifacts,Stable model-artifact layout and verification ...
2,hf_finetuning_lab.config,
3,hf_finetuning_lab.data,
4,hf_finetuning_lab.data.hub,Load and normalize Hugging Face Hub text-class...
5,hf_finetuning_lab.evaluation,
6,hf_finetuning_lab.evaluation.metrics,
7,hf_finetuning_lab.evaluation.robust,"Robust evaluation utilities: calibration, thre..."
8,hf_finetuning_lab.experiments,"Experiment-tracking utilities (run IDs, datase..."
9,hf_finetuning_lab.token_classification,"Token-classification (NER) utilities: schema, ..."


## 5) Notebook Index

The nine notebooks ship together as a guided tour. Each one is independent (synthetic data + offline smoke) but they all build on the same modules.

In [10]:
notebook_dir = ROOT / 'notebooks'
ordered = sorted(p.name for p in notebook_dir.glob('*.ipynb'))
pd.DataFrame({'notebook': ordered})

,notebook
0,01_hf_text_classification_workflow.ipynb
1,02_experiment_management.ipynb
2,03_robust_evaluation.ipynb
3,04_hub_datasets.ipynb
4,05_token_classification.ipynb
5,06_semantic_search.ipynb
6,07_governance_template.ipynb
7,08_serving_hardening.ipynb
8,09_v1_capstone.ipynb


## 6) v1.0 Release Checklist
- [x] Stable CLI surface: enumerated by `hf-lab list-commands`, version surfaced by `hf-lab version`.
- [x] Stable model artifact layout (`required` + `alternative-required` + `recommended` tiers) with `verify_artifact` + `hf-lab verify-artifact --strict` gating.
- [x] CPU smoke coverage: every notebook + the test suite (`pytest`) runs without a model download.
- [ ] GPU smoke coverage: training paths in `hf_finetuning_lab.training` are documented but only manually exercised on GPU hosts. Wire a GPU CI job when one is available.
- [x] Documentation refresh: `docs/architecture.md` lists the v1.0 module map and the artifact contract.
- [x] Versioning: `pyproject.toml` and `hf_finetuning_lab.__version__` both at `1.0.0`.
- For production: ship the artifact through a model registry, run `hf-lab verify-artifact --strict` in the deployment pipeline, and gate promotion on a clean `notebooks/07_governance_template.ipynb` checklist.